# Hafta 7 · Sinyal: Zaman Taşıyan Veri — EKG ile Filtreleme, Spektrum, Öznitelik ve Dürüst Doğrulama

**SVY 5430 Biyomedikal Verilerin Analizi ve Yapay Zeka** · Fırat Üniversitesi Yazılım ve Bilişim Araştırma Enstitüsü

Veri: PhysioNet **MIT-BIH Arrhythmia** veri tabanından gerçek EKG kayıtları (`wfdb` ile doğrudan indirilir; her atım kardiyolog tarafından etiketli).
Beş bölüm: sinyale bakmak ve Nyquist, filtreleme, spektrum, R tepeleri ve HRV öznitelikleri, atım sınıflandırmasında kayıt-bazlı doğrulama.
BVYZ-03 ve BVYZ-04 projeleri bu kodu doğrudan kullanabilir.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

KAYITLAR = ["100", "101", "103", "106", "119", "200", "208", "223"]   # normal ağırlıklı + ektopik atımı bol kayıtlar
try:
    import wfdb
    kayit = {}
    for r in KAYITLAR:
        rec = wfdb.rdrecord(r, pn_dir="mitdb"); ann = wfdb.rdann(r, "atr", pn_dir="mitdb")
        kayit[r] = dict(x=rec.p_signal[:, 0], fs=rec.fs, r_idx=ann.sample, sembol=np.array(ann.symbol))
    fs = kayit["100"]["fs"]; print(f"{len(kayit)} kayıt indirildi · örnekleme hızı {fs} Hz · 30 dk/kayıt")
except Exception as e:
    print("İndirme başarısız, sentetik EKG üretiliyor:", e)
    fs = 360; rng = np.random.default_rng(0); kayit = {}
    for i, r in enumerate(KAYITLAR):
        t = np.arange(0, 600, 1 / fs); x = np.zeros_like(t); hr = 60 + 10 * i
        rr = np.cumsum(rng.normal(60 / hr, 0.03, int(600 * hr / 60)))
        r_idx = (rr[rr < 600] * fs).astype(int)
        for k in r_idx: x += np.exp(-((t - k / fs) / 0.012) ** 2) + 0.25 * np.exp(-((t - k / fs - 0.3) / 0.06) ** 2)
        sembol = rng.choice(["N", "V"], len(r_idx), p=[0.9, 0.1])
        kayit[r] = dict(x=x + 0.05 * rng.standard_normal(t.size) + 0.3 * np.sin(2 * np.pi * 50 * t), fs=fs, r_idx=r_idx, sembol=sembol)

## 1 · Sinyale bakmak

Saniyede 360 ölçüm. Beş saniyelik pencerede yaklaşık altı atım görmeliyiz. Örnekleme hızı neden önemli? Saniyede 360 fotoğraf çekiyorsanız
saniyede en fazla 180 kez titreyen bir şeyi yakalayabilirsiniz; bunun üstü yanlış frekansa "katlanır" (Nyquist). EKG için 360 fazlasıyla yeter.

In [ ]:
x = kayit["100"]["x"]; t = np.arange(len(x)) / fs
plt.figure(figsize=(12, 3)); plt.plot(t[:5*fs], x[:5*fs], lw=.9, c="#0b1526"); plt.xlabel("saniye"); plt.ylabel("mV"); plt.title("Kayıt 100 · ilk 5 saniye (ham)"); plt.tight_layout()

## 2 · Filtreleme: gürültüyü süzmek

EKG'de iki tipik gürültü var: solunum ve hareketten gelen yavaş taban çizgisi kayması (< 0.5 Hz) ve şebeke uğultusu (50 Hz).
Bant geçiren filtre 0.5–40 Hz arasını bırakır, gerisini süzer. `filtfilt` ileri-geri uygulandığı için sinyali kaydırmaz.

In [ ]:
def bant_gecir(x, fs, lo=0.5, hi=40, n=4):
    b, a = signal.butter(n, [lo, hi], btype="band", fs=fs)
    return signal.filtfilt(b, a, x)

# kasıtlı olarak gürültü ekleyip süzüyoruz, ki fark görünsün
rng = np.random.default_rng(1)
kirli = x + 0.3 * np.sin(2 * np.pi * 50 * t) + 0.4 * np.sin(2 * np.pi * 0.2 * t)
temiz = bant_gecir(kirli, fs)
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(t[:5*fs], kirli[:5*fs], lw=.8, c="#6b7a92"); ax[0].set_title("Şebeke (50 Hz) + taban çizgisi kayması eklenmiş")
ax[1].plot(t[:5*fs], temiz[:5*fs], lw=.9, c="#0f9b8e"); ax[1].set_title("0.5–40 Hz bant geçiren filtre sonrası"); ax[1].set_xlabel("saniye")
plt.tight_layout()

## 3 · Spektrum: sinyali notalarına ayırmak

Welch yöntemi, sinyalde hangi frekansın ne kadar güç taşıdığını gösterir. 50 Hz'deki sivri çizgi şebekedir; filtre sonrası kaybolmalı.
EEG'de aynı grafik delta/teta/alfa/beta bantlarını, HRV'de LF/HF oranını verir.

In [ ]:
f1, P1 = signal.welch(kirli, fs=fs, nperseg=4096); f2, P2 = signal.welch(temiz, fs=fs, nperseg=4096)
plt.figure(figsize=(9, 3.6)); plt.semilogy(f1, P1, c="#6b7a92", label="kirli"); plt.semilogy(f2, P2, c="#0f9b8e", label="filtreli")
plt.axvline(50, ls="--", c="#be3a5a", lw=1, label="50 Hz"); plt.xlim(0, 80); plt.xlabel("Hz"); plt.ylabel("güç"); plt.legend(); plt.title("Güç spektral yoğunluğu (Welch)"); plt.tight_layout()

## 4 · R tepeleri, RR aralıkları ve HRV

Atımı bulmak için R tepesini ararız. Burada iki yol var: kendimiz tepe bulmak (`find_peaks`) ya da kardiyologun anotasyonunu kullanmak.
İkisini karşılaştırıyoruz. Ardışık R tepeleri arasındaki süre (RR) kalp hızı değişkenliğini (HRV) verir: RMSSD, pNN50 gibi sayılar.

In [ ]:
temiz100 = bant_gecir(x, fs)
tepe, _ = signal.find_peaks(temiz100, distance=int(0.4 * fs), height=np.percentile(temiz100, 98) * 0.5)
r_ann = kayit["100"]["r_idx"]; r_ann = r_ann[(r_ann > 0) & (r_ann < len(x))]
print(f"find_peaks: {len(tepe)} tepe · anotasyon: {len(r_ann)} atım  (30 dakika)")

plt.figure(figsize=(12, 3)); seg = slice(0, 5 * fs)
plt.plot(t[seg], temiz100[seg], lw=.9, c="#0b1526")
plt.plot(t[tepe[tepe < 5*fs]], temiz100[tepe[tepe < 5*fs]], "v", c="#be3a5a", label="find_peaks")
plt.plot(t[r_ann[r_ann < 5*fs]], temiz100[r_ann[r_ann < 5*fs]] + 0.15, "^", c="#0f9b8e", label="kardiyolog anotasyonu")
plt.legend(); plt.xlabel("saniye"); plt.title("R tepeleri"); plt.tight_layout()

rr = np.diff(r_ann) / fs * 1000                      # ms
hrv = {"ortalama RR (ms)": rr.mean(), "kalp hızı (bpm)": 60000 / rr.mean(), "SDNN (ms)": rr.std(),
       "RMSSD (ms)": np.sqrt(np.mean(np.diff(rr) ** 2)), "pNN50 (%)": 100 * np.mean(np.abs(np.diff(rr)) > 50)}
pd.Series(hrv).round(1)

## 5 · Atım sınıflandırması ve dürüst doğrulama

Her atım için bir pencere (R tepesinin 90 ms öncesi, 160 ms sonrası) ve birkaç RR özniteliği çıkarıyoruz. Hedef: normal (N) mi, ventriküler
ektopik (V) mi? Sonra 2. haftanın sorusunu soruyoruz: pencereleri rastgele bölersek aynı kaydın atımları iki kümeye düşer. **Kayıt-bazlı**
bölünce ne oluyor?

In [ ]:
def atim_oznitelik(kayit, r, once=0.09, sonra=0.16):
    x = bant_gecir(kayit["x"], kayit["fs"]); fs_ = kayit["fs"]; n1, n2 = int(once * fs_), int(sonra * fs_)
    r_idx, sem = kayit["r_idx"], kayit["sembol"]
    rows, ys, gr = [], [], []
    for i in range(2, len(r_idx) - 2):
        if sem[i] not in ("N", "V"): continue
        k = r_idx[i]
        if k - n1 < 0 or k + n2 >= len(x): continue
        pencere = x[k - n1 : k + n2]
        rr_on, rr_son = (r_idx[i] - r_idx[i-1]) / fs_, (r_idx[i+1] - r_idx[i]) / fs_
        rr_yerel = np.mean(np.diff(r_idx[i-2:i+3])) / fs_
        rows.append(np.concatenate([pencere[::3], [rr_on, rr_son, rr_on / rr_yerel, pencere.max() - pencere.min(), np.abs(np.diff(pencere)).mean()]]))
        ys.append(1 if sem[i] == "V" else 0); gr.append(r)
    return np.array(rows), np.array(ys), np.array(gr)

parcalar = [atim_oznitelik(kayit[r], r) for r in kayit]
Xb = np.vstack([p[0] for p in parcalar]); yb = np.concatenate([p[1] for p in parcalar]); gb = np.concatenate([p[2] for p in parcalar])
print(f"{len(yb)} atım · V oranı %{100*yb.mean():.1f} · kayıt sayısı {len(set(gb))}")
pd.Series(yb).groupby(gb).agg(["count", "mean"]).rename(columns={"count": "atım", "mean": "V oranı"}).round(3).T

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0, n_jobs=-1)
rastgele = cross_val_score(rf, Xb, yb, cv=StratifiedKFold(4, shuffle=True, random_state=0), scoring="f1").mean()
grup     = cross_val_score(rf, Xb, yb, cv=GroupKFold(4), groups=gb, scoring="f1").mean()
print(f"Rastgele bölme (aynı kayıt iki kümede): F1 = {rastgele:.3f}")
print(f"Kayıt-bazlı bölme (GroupKFold):         F1 = {grup:.3f}   <- yeni hastaya genelleme")

**Okuyun:** Fark ne kadar? Rastgele bölmede model her hastanın "normal" atımının şeklini ezberleyip V'yi ona göre ayırıyor. Yeni bir hastaya
geçince o şablon yok; başarım düşüyor. Makalelerde "atım-içi" (intra-patient) ile "hasta-arası" (inter-patient) sonuçlar bu yüzden ayrı raporlanır.

> **Deneyin:** `KAYITLAR` listesine "207" ve "233" ekleyin (zor kayıtlar). Kayıt-bazlı F1 ne oldu? Öznitelik olarak yalnızca RR oranlarını bırakıp
> pencereyi çıkarın; hangi öznitelikler yeni hastaya daha iyi genelliyor?

## 6 · Bugün ne gördük?

| Adım | Araç | Alışkanlık |
|---|---|---|
| Örnekleme ve Nyquist | `fs`, sinyalin yarısı | Hangi frekansı görebildiğinizi bilin |
| Filtreleme | Butterworth bant geçiren, `filtfilt` | Önce/sonra spektrumunu çizin |
| Spektrum | Welch PSD | 50 Hz çizgisi gitti mi? EEG bantları, HRV LF/HF |
| Öznitelik | R tepe, RR, HRV, pencere | Pencere + RR oranları birlikte |
| Doğrulama | GroupKFold(kayıt) | Hasta-arası sonuç raporlanır |

**Projeler:** BVYZ-03 bu defterin Bölüm 5'ini doğrudan genişletebilir (AAMI sınıfları, makro-F1). BVYZ-04 Bölüm 2–3'ü PPG'ye uyarlar.
BVYZ-08 Bölüm 3'ü EEG bantlarına taşır.
